In [ ]:
from woa import WOA
import pandas as pd

Import the datasets from the csv file (selected date is 15.07.2025):
- energy  prices
- load
- pv generation

In [ ]:
prices_csv = pd.read_csv("entsoe_datasets/energy_prices_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
prices_csv = prices_csv[prices_csv["Sequence"] == "Sequence 1"]
price_series = []
for price in prices_csv.values:
    price_series.append(price[1])

pv_generation_csv = pd.read_csv("entsoe_datasets/pv_generation_15_07_2025.csv",
                            parse_dates=["time"],
                            index_col=["time"],
                            usecols=lambda col: "time" in col or "Day-ahead" in col,
                            date_format="%Y-%m-%d %H:%M:%S")
print(pv_generation_csv)
normalized_pv = pv_generation_csv.values / (pv_generation_csv.values).max()
pv_series = []
for pv in normalized_pv:
    pv_series.append(pv[0])
print(pv_series)

load_csv = pd.read_csv("entsoe_datasets/load_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
print(load_csv)
normalized_load = (load_csv.values) / (load_csv.values).max()
load_series = []
for load in normalized_load:
    load_series.append(load[0])
print(load_series)


Define IEEE-33 load profiles

In [32]:
load_data = [
    (100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]
E_load = []
for (p, q) in enumerate(load_data):
    E_load.append(p * load_series[0])
print(E_load)

[np.float64(0.0), np.float64(0.7341149994868712), np.float64(1.4682299989737424), np.float64(2.202344998460614), np.float64(2.936459997947485), np.float64(3.670574997434356), np.float64(4.404689996921228), np.float64(5.138804996408099), np.float64(5.87291999589497), np.float64(6.607034995381841), np.float64(7.341149994868712), np.float64(8.075264994355583), np.float64(8.809379993842455), np.float64(9.543494993329325), np.float64(10.277609992816197), np.float64(11.011724992303067), np.float64(11.74583999178994), np.float64(12.479954991276811), np.float64(13.214069990763681), np.float64(13.948184990250553), np.float64(14.682299989737423), np.float64(15.416414989224295), np.float64(16.150529988711167), np.float64(16.884644988198037), np.float64(17.61875998768491), np.float64(18.35287498717178), np.float64(19.08698998665865), np.float64(19.82110498614552), np.float64(20.555219985632394), np.float64(21.289334985119265), np.float64(22.023449984606135), np.float64(22.75756498409301)]


Temporary definition of global variables - they will be extracted based on the datatset.

In [ ]:
w1 = w2 = w3 = 1
buy_price = sell_price = price_series[0]
E_pv = pv_series[0]
target_energy = 3715 # MW of active power in a standard IEEE-33 bus system
e_bat = 1
eff_charge = 0.9
eff_discharge = 0.9
k_slope = 1
num_household = 33
E_loss = 0.1
E_bat = currSoC = prevSoC = [1 for i in range(num_household)]
batt_cost = batt_capacity = [2 for i in range(num_household)]
maxSoC = [95 for i in range(num_household)]
minSoC = [10 for i in range(num_household)]
l1 = l2 = l3 = l4 = 1
grid_energy_min = -100
grid_energy_max = 300

Compute the energy exchange between a household and the grid.

In [ ]:
def getGridForHousehold(houseIdx):
    E_i_grid = 0
    E_i_grid = E_load[houseIdx] - E_pv[houseIdx] + E_bat[houseIdx]
    return E_i_grid

Compute the total exchange with the grid, for the entire VPP.

In [ ]:
def getTotalGridEnergy():
    E_grid = 0
    for i in range(num_household):
        E_grid += getGridForHousehold(i)
    E_grid += E_loss
    return E_grid

Get the state of charge for the next time interval.

In [ ]:
def computeSoCForBat(idx, e_bat):
    currSoC[idx] = 0
    if (e_bat >= 0):
        currSoC[idx] = prevSoC[idx] + (eff_charge * e_bat)
    else:
        currSoC[idx] = prevSoC[idx] - (eff_discharge * abs(e_bat))

def computeTotalSoC(num_household, e_bat_lst):
    for i in range(num_household):
        computeSoCForBat(i, e_bat_lst[i])

Compute the cost for the current time interval.

In [ ]:
def computeCost(grid_energy):
    cost = 0
    if (grid_energy >= 0):
        cost = grid_energy * buy_price
    else:
        cost = abs(grid_energy) * (sell_price) * (-1)
    return cost 

Compute the tracking component for the current time interval.

In [ ]:
def computeTrackComponent(grid_energy):
    track = pow((grid_energy - target_energy), 2)  
    return track

Compute the battery penalty function, based on the current and previous SoC.

In [ ]:
def computeBattFctForBatt(idx):
    batt_i = abs(k_slope / 100) * ((prevSoC[idx] - currSoC[idx]) / batt_capacity[idx]) * batt_cost[idx]
    return batt_i

def computeTotalBattFct(num_household):
    F_bat = 0
    for i in range(num_household):
        F_bat += computeBattFctForBatt(i)
    return F_bat


The function computeBattPenaltyForBatt computes the penalty for a battery. Apply it to all the batteries to compute the total penalty applied to the batteries.

In [ ]:
def computeBattPenaltyForBatt(currSoC, maxSoC, minSoC):
    F_pen_bat_i = l1* pow(max(0, (currSoC - maxSoC)), 2) + l2 *pow(max(0, (minSoC - currSoC)), 2) 
    return F_pen_bat_i

def computeTotalBattPenalty(num_household):
    F_pen_bat = 0
    for i in range(num_household):
        F_pen_bat += computeBattPenaltyForBatt(currSoC[i], maxSoC[i], minSoC[i])
    return F_pen_bat

Compute the grid penalty component.

In [ ]:
def computeGridPenalty(E_grid, E_grid_max, E_grid_min):
    F_pen_grid = l3 * pow(max(0, (E_grid - E_grid_max)), 2) + l4 * pow(min(0, (E_grid - E_grid_min)), 2)
    return F_pen_grid

Using the 2 previous components, compute the penalty function, for the current timestep, for the VPP.

In [ ]:
def computePenalty(num_household, E_grid, E_grid_max, E_grid_min):
    F_pen_bat = computeTotalBattPenalty(num_household)
    F_pen_grid = computeGridPenalty(E_grid, E_grid_max, E_grid_min)
    F_pen = F_pen_bat + F_pen_grid
    return F_pen

Get the fitness function for an individual

In [ ]:
def fitness():
    grid_energy = getTotalGridEnergy()
    f_cost = computeCost(grid_energy)
    f_track = computeTrackComponent(grid_energy)
    f_bat = computeTotalBattFct(num_household)
    f_pen = computePenalty(num_household, grid_energy, grid_energy_max, grid_energy_min)
    fitness = w1 * f_cost + w2 * f_track + w3 * f_bat + f_pen
    return fitness

In [ ]:
woa = WOA(100, 500, -500, fitness)


In [ ]:
best = woa.computeBest(100, num_household, 500, -500, 100)
print(best.fitness_value)
print(best.transf_energy)